In [72]:
import torch
from torch import nn
from torch import functional as F

X=torch.rand(1,20)
X

tensor([[0.8096, 0.5383, 0.4547, 0.9201, 0.0796, 0.3690, 0.3045, 0.6088, 0.3580,
         0.5710, 0.9621, 0.4728, 0.7499, 0.6990, 0.9993, 0.1363, 0.4118, 0.8198,
         0.3381, 0.7507]])

In [73]:

class mysequential(nn.Module):
    def __init__(self,*args) -> None:
        super().__init__()
        for block in args:
            self._modules[block]=block
        self.blocks=args
    
    def __getitem__(self,i):#重载运算符
            return self.blocks[i]
    
    def forward(self,X):#自定义,直接调用forward
        for b in self._modules.values():
            X=b(X)
        return X
    
    def named_parameters(self):
        ans=[]
        for block in self.blocks:
            try:
                ans.append([block.weight,block.bias])
                
            except AttributeError:#无属性,如ReLU
                pass
            #'ReLU' object has no attribute 'weight'
        return ans

net=mysequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)


tensor([[-0.0406, -0.1649,  0.1261, -0.1731,  0.0989,  0.0083,  0.0603,  0.1996,
          0.0188,  0.4377]], grad_fn=<AddmmBackward0>)

In [74]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.0138,  0.0094, -0.0323,  ...,  0.0389, -0.0297, -0.0056],
                      [-0.0294, -0.0063, -0.0391,  ..., -0.0574,  0.0055,  0.0179],
                      [ 0.0330, -0.0445,  0.0177,  ..., -0.0057,  0.0014, -0.0033],
                      ...,
                      [ 0.0343, -0.0222,  0.0181,  ..., -0.0212, -0.0573,  0.0341],
                      [-0.0253, -0.0417,  0.0049,  ...,  0.0146,  0.0358,  0.0233],
                      [ 0.0283, -0.0254,  0.0235,  ..., -0.0317, -0.0232,  0.0297]])),
             ('bias',
              tensor([ 0.0390, -0.0596,  0.0226, -0.0212, -0.0280, -0.0400, -0.0367,  0.0267,
                      -0.0118,  0.0625]))])

In [75]:
net[0].weight

Parameter containing:
tensor([[-0.1696,  0.0962, -0.1278,  ..., -0.1975, -0.1989,  0.0420],
        [ 0.1692,  0.0638, -0.0404,  ..., -0.1300, -0.0952,  0.0956],
        [ 0.0821, -0.2156, -0.0649,  ..., -0.0763, -0.1868, -0.0171],
        ...,
        [ 0.0686, -0.0868,  0.1234,  ...,  0.0106, -0.1028, -0.0978],
        [-0.1499, -0.1923,  0.0223,  ...,  0.0231,  0.1958,  0.0265],
        [ 0.0240, -0.2037, -0.0596,  ...,  0.1737,  0.1513, -0.2072]],
       requires_grad=True)

In [76]:
print(*[par for par in net.named_parameters()])

[Parameter containing:
tensor([[-0.1696,  0.0962, -0.1278,  ..., -0.1975, -0.1989,  0.0420],
        [ 0.1692,  0.0638, -0.0404,  ..., -0.1300, -0.0952,  0.0956],
        [ 0.0821, -0.2156, -0.0649,  ..., -0.0763, -0.1868, -0.0171],
        ...,
        [ 0.0686, -0.0868,  0.1234,  ...,  0.0106, -0.1028, -0.0978],
        [-0.1499, -0.1923,  0.0223,  ...,  0.0231,  0.1958,  0.0265],
        [ 0.0240, -0.2037, -0.0596,  ...,  0.1737,  0.1513, -0.2072]],
       requires_grad=True), Parameter containing:
tensor([ 0.1335,  0.1975, -0.1133,  0.2146,  0.1939,  0.1820, -0.1557,  0.1318,
         0.1880,  0.0278, -0.1091, -0.0154, -0.0959, -0.0785,  0.0937, -0.1715,
        -0.0409, -0.1047, -0.1171, -0.1902, -0.0748,  0.1650, -0.0201,  0.1705,
         0.0020, -0.1281,  0.0132, -0.1385,  0.1496,  0.0097, -0.1604, -0.0218,
         0.0858,  0.0679, -0.0627,  0.0836,  0.1535,  0.1984, -0.0334, -0.0736,
         0.1162, -0.1112,  0.1861, -0.2008,  0.1504,  0.0110, -0.0225,  0.2075,
        -0.15

In [77]:
net2=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net2(X)#默认做法

tensor([[-0.0035,  0.1577, -0.0996,  0.1555, -0.0299,  0.0619,  0.1045,  0.1327,
         -0.0196,  0.0014]], grad_fn=<AddmmBackward0>)

In [78]:
print(*[(name,par.shape) for name,par in net2.named_parameters()])

('0.weight', torch.Size([256, 20])) ('0.bias', torch.Size([256])) ('2.weight', torch.Size([10, 256])) ('2.bias', torch.Size([10]))


In [79]:
net2.state_dict()['2.bias']#通过序号访问

tensor([-0.0038, -0.0607, -0.0227, -0.0463, -0.0551,  0.0570,  0.0431,  0.0273,
        -0.0337, -0.0474])